In [0]:
import logging

logger = logging.getLogger("bronze-parameters")
logger.setLevel(logging.INFO)

dbutils.widgets.text("storage_account_name", "")
dbutils.widgets.text("source_container", "")
dbutils.widgets.text("target_container", "")
dbutils.widgets.text("table_names", "")

storageAccountName = dbutils.widgets.get("storage_account_name").strip()
sourceContainerName = dbutils.widgets.get("source_container").strip()
targetContainerName = dbutils.widgets.get("target_container").strip()
tableNamesInput = dbutils.widgets.get("table_names").strip()

if not storageAccountName:
    storageAccountName = "datalakezakariae2026"

if not sourceContainerName:
    sourceContainerName = "source"

if not targetContainerName:
    targetContainerName = "bronze"

sourceRootPath = (
    f"abfss://{sourceContainerName}@{storageAccountName}.dfs.core.windows.net/"
)

if sourceRootPath == "abfss://@.dfs.core.windows.net/":
    raise ValueError(
        "Invalid sourceRootPath. storage_account_name and source_container are empty."
    )

excludedFolderNames = {
    "_temporary",
    "_schemas",
    "schema",
    "schemas",
    "checkpoint",
    "checkpoints"
}

if tableNamesInput:
    fileNames = [
        tableName.strip()
        for tableName in tableNamesInput.split(",")
        if tableName.strip()
    ]
else:
    fileNames = [
        fileInfo.name.rstrip("/")
        for fileInfo in dbutils.fs.ls(sourceRootPath)
        if fileInfo.name.endswith("/")
        and fileInfo.name.rstrip("/") not in excludedFolderNames
    ]

datasets = [
    {
        "file_name": fileName,
        "storage_account_name": storageAccountName,
        "source_container": sourceContainerName,
        "target_container": targetContainerName
    }
    for fileName in fileNames
]

if not datasets:
    raise ValueError(
        f"No datasets found in source path: {sourceRootPath}"
    )

dbutils.jobs.taskValues.set(
    key="output_dataset",
    value=datasets
)

logger.info("Generated %s dataset parameters", len(datasets))

datasets

[{'file_name': 'customers',
  'storage_account_name': 'datalakezakariae2026',
  'source_container': 'source',
  'target_container': 'bronze'},
 {'file_name': 'orders',
  'storage_account_name': 'datalakezakariae2026',
  'source_container': 'source',
  'target_container': 'bronze'},
 {'file_name': 'products',
  'storage_account_name': 'datalakezakariae2026',
  'source_container': 'source',
  'target_container': 'bronze'},
 {'file_name': 'regions',
  'storage_account_name': 'datalakezakariae2026',
  'source_container': 'source',
  'target_container': 'bronze'}]